# Memory 

- Till now we have seen the agent can get the answer what we need by invoking the LLM.

- Lets test whether it can have a memory or can keep track of the interactions we made

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

chat_model = ChatOpenAI(
    model="local-llm",
    base_url="http://localhost:8080/v1",
    temperature=0.7
)

agent = create_agent(
    model = chat_model
)

In [ ]:
## Lets invoke agent

from langchain.messages import HumanMessage
from pprint import pprint

print("## ====== First message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of India?")]
})
pprint(response["messages"])


print("## ====== Second message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of Telanga?")]
})
pprint(response["messages"])

In [ ]:
# Lets check whether agent can remember the first message.

response = agent.invoke({
    "messages": [HumanMessage("What is my first question I have asked you?")]
})
pprint(response["messages"][-1])

### From above response we can say that it does not have memory of the previous messages.

- Lets give the agent memory to keep track of messages.


In [ ]:
## one of the way is to keep track of messages
from langchain.messages import AIMessage
response = agent.invoke(
    {
    "messages": [HumanMessage("What is capital of India?"),
                 AIMessage("The capital of India is **New Delhi**."),
                 HumanMessage("What is capital city of Telanga?"),
                 AIMessage("The capital of the state of Telangana is **Hyderabad**."),
                 HumanMessage("What is my first question I have asked you?")]
}
)

pprint(response["messages"][-1])

### Does above solution scalable?
- Answer is no 😔, why because we have keep track of all the message manually or runtime saving

### Do we have scalable solution?
- Answer is yes 🚀

In [ ]:
## First we need to configure our agent with memory.
from langgraph.checkpoint.memory import InMemorySaver
agent = create_agent(
    model=chat_model,
    checkpointer= InMemorySaver()
)

In [ ]:
## Lets invoke agent and see whether it can remember the first message


print("## ====== First message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of India?")]
})
pprint(response["messages"])


print("## ====== Second message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of Telanga?")]
})
pprint(response["messages"])

### Why do we see error in above cell excution 🤔...?

In [ ]:
# lets add config while invoking the agent and check we whether it can remember the message
config = { "configurable": {"thread_id": 1} }

print("## ====== First message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of India?")]
},
config = config
)
pprint(response["messages"])


print("\n ## ====== Second message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is capital of Telanga?")]
},
config = config
)
pprint(response["messages"])

print("\n ## ====== Third message ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("What is my first question I have asked you?")]
},
config = config
)
pprint(response["messages"])

In [ ]:
# Lets have one more thread, which is equivalent to new session.
config = { "configurable": {"thread_id": 2} }

print("## ====== First response from agent ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("Hello, I'm groot. I'm a fictional character. I look like a tree.")]
},
config = config
)
pprint(response["messages"][-1])


print("\n ## ====== Second response from agent ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("How do I look like?")]
},
config = config
)
pprint(response["messages"][-1])

print("\n ## ====== Third response from agent ====== ##")
response = agent.invoke({
    "messages": [HumanMessage("I'm a real person?")]
},
config = config
)
pprint(response["messages"][-1])

In [ ]:
# How to acces different sessions or threads?
config = { "configurable": {"thread_id": 1} }

response = agent.invoke({
    "messages": [HumanMessage("Give me summary of our chat")]
},
config = config
)
print("## ====== Summary of first conversation ====== ##")
pprint(response["messages"][-1])

config = { "configurable": {"thread_id": 2} }

response = agent.invoke({
    "messages": [HumanMessage("Give me summary of our chat")]
},
config = config
)
print("## ====== Summary of second conversation ====== ##")
pprint(response["messages"][-1])